# 0.0.0 Master corpus build

This notebook builds a single full cleaned mention table for dark matter from the ADS source parquet.

The goal is architectural, not methodological: centralize normalization, filtering, cleaning, and mention extraction in one place so downstream corpora can be *derived* from a canonical master artifact rather than rebuilt independently.

### Scope

This notebook builds the upstream master mention inventory for the project. Its purpose is to host a single validated, uncapped, cleaned mention table that downstream corpus and analysis notebooks can derive from.

---

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

CODE_DIR = WORKSPACE_DIR / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)


### Normalization
Cleaning, filtering, and extraction settings are centralized here for methodological consistency and easier maintenance.


In [ ]:
import importlib

import dm_term_normalization.normalization as normalization
importlib.reload(normalization)

from dm_term_normalization.normalization import (
    TARGET,
    pat,
    MAX_CTX_WORDS,
    MIN_CTX_WORDS,
    MIN_CENTER_SENT_WORDS,
    SOFT_CTX_WORDS,
    MAX_DM_PER_CTX,
    MAX_MATH_TOKENS_PER_CTX,
    canon_dm,
    iter_sentences_norm,
)

MASTER_RUN_ID = 'master_corpus'
PARQUET_PATH = DATA_DIR / 'parquet' / 'ADS_MAIN_FRAME_2025.parquet'

master_root = LOCAL_DIR / 'corpora' / MASTER_RUN_ID
master_root.mkdir(parents=True, exist_ok=True)

MASTER_PARQUET = master_root / 'master_corpus.parquet'
MASTER_MANIFEST = master_root / 'master_corpus_manifest.json'

FORCE_REBUILD_MASTER = True

BIN_YEARS = 5
ANALYSIS_START_YEAR = 1986
END_YEAR = 2025
EARLIEST_BACKFILL_YEAR = 1975


def workspace_relative(path: Path) -> str:
    return str(path.resolve().relative_to(WORKSPACE_DIR)).replace("\\", "/")


print('PARQUET_PATH:', PARQUET_PATH)
print('MASTER_PARQUET:', MASTER_PARQUET)
print('MASTER_MANIFEST:', MASTER_MANIFEST)
print('FORCE_REBUILD_MASTER:', FORCE_REBUILD_MASTER)


### Load and filter source records
This enforces a high-level filtering logic before mention extraction.


In [ ]:
import pandas as pd
import numpy as np
import json
import hashlib
import re
from datetime import datetime, timezone
from tqdm.auto import tqdm

if MASTER_PARQUET.exists() and MASTER_MANIFEST.exists() and not FORCE_REBUILD_MASTER:
    master_df = pd.read_parquet(MASTER_PARQUET)
    with open(MASTER_MANIFEST, 'r', encoding='utf-8') as f:
        master_manifest = json.load(f)
    print(f'Loaded cached master corpus -> {MASTER_PARQUET}')
    print(f'Loaded cached manifest -> {MASTER_MANIFEST}')
else:
    df = pd.read_parquet(PARQUET_PATH)
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    mask = df['year'].notna() & df['abstract'].notna()
    df = df.loc[mask].assign(abstract=df.loc[mask, 'abstract'].str.lower()).copy()
    df['year'] = df['year'].astype(int)

    bad_arx = [
        'quantitative finance',
        'quantitative biology',
        'economics',
        'electrical engineering and systems science',
    ]
    bad_doc = [
        'pressrelease', 'mastersthesis', 'misc', 'techreport', 'dataset', 'proposal', 'proceedings',
        'abstract', 'newsletter', 'obituary', 'talk', 'circular', 'bookreview', 'software',
        'editorial', 'erratum', 'phdthesis',
    ]

    df = df[~df['arxiv_category'].isin(bad_arx)].copy()
    df = df[~df['doctype'].isin(bad_doc)].copy()

    biology_terms_strict = {
        'microbial', 'bacterial', 'genome', 'genomes', 'eukaryotic', 'genomic',
        'microbes', 'microorganisms', 'bacteria', 'metabolites', 'phyla', 'mags', 'metagenomics'
    }
    biology_pattern = re.compile(r"\b(?:%s)\b" % '|'.join(map(re.escape, biology_terms_strict)), flags=re.I)
    df = df.loc[~df['abstract'].astype('string').str.contains(biology_pattern, na=False)].copy()

    df = df.loc[df['year'].between(EARLIEST_BACKFILL_YEAR, END_YEAR, inclusive='both')].copy()

    df = (
        df.assign(
            bibcode=df['bibcode'].astype('string'),
            abstract=df['abstract'].astype('string'),
        )
        .reset_index(drop=False)
        .rename(columns={'index': 'source_row_index'})
    )

    print('Filtered source rows:', len(df))
    df.head(3)


### Mention Extraction Helpers


In [ ]:
def assign_time_bin(year: int) -> str | None:
    if year > END_YEAR or year < ANALYSIS_START_YEAR:
        return None
    right_edge = ANALYSIS_START_YEAR + BIN_YEARS - 1
    left_edge = ANALYSIS_START_YEAR
    while year > right_edge and right_edge < END_YEAR:
        left_edge += BIN_YEARS
        right_edge += BIN_YEARS
    return f'{left_edge}-{min(right_edge, END_YEAR)}'

def build_examples_from_abstract_pm1(abstract: str, bibcode: str = '') -> list[dict]:
    sents = list(iter_sentences_norm(abstract))
    out = []
    seen_ctx = set()

    for i, s in enumerate(sents):
        matches = list(pat.finditer(s))
        if not matches:
            continue

        m = matches[0]
        mention_order = len(out)

        left = (sents[i - 1] + ' ') if i - 1 >= 0 else ''
        right = (' ' + sents[i + 1]) if i + 1 < len(sents) else ''
        ctx = left + s + right
        start = len(left) + m.start()
        end = len(left) + m.end()

        center_sent_words = len(s.split())
        n_dm_ctx = len(pat.findall(ctx))
        n_math_ctx = len(re.findall(r'\bMATH_TOKEN\b', ctx))
        is_listy = (
            len(re.findall(r"\(\s*[ivx]+\s*\)|\(\s*[a-z]\s*\)|\b[1-9]\)", ctx, flags=re.I)) >= 2
            or ctx.count(';') >= 3
            or '*' in ctx
        )

        if (
            len(ctx.split()) > SOFT_CTX_WORDS
            or n_dm_ctx > MAX_DM_PER_CTX
            or n_math_ctx > MAX_MATH_TOKENS_PER_CTX
            or is_listy
        ):
            ctx = s
            start, end = m.start(), m.end()
            n_math_ctx = len(re.findall(r'\bMATH_TOKEN\b', ctx))
            is_listy = (
                len(re.findall(r"\(\s*[ivx]+\s*\)|\(\s*[a-z]\s*\)|\b[1-9]\)", ctx, flags=re.I)) >= 2
                or ctx.count(';') >= 3
                or '*' in ctx
            )

        if len(ctx.split()) > MAX_CTX_WORDS:
            continue
        if len(ctx.split()) < MIN_CTX_WORDS:
            continue
        if center_sent_words < MIN_CENTER_SENT_WORDS:
            continue
        if n_math_ctx > MAX_MATH_TOKENS_PER_CTX:
            continue
        if is_listy and len(ctx.split()) > 120:
            continue
        if canon_dm(ctx[start:end]) != TARGET:
            continue

        ctx = re.sub(r'\s+', ' ', ctx).strip()
        dedupe_key = (bibcode, ctx)
        if dedupe_key in seen_ctx:
            continue
        seen_ctx.add(dedupe_key)

        out.append({
            'sent': ctx,
            'start': int(start),
            'end': int(end),
            'bibcode': bibcode,
            'has_math': bool(re.search(r'\bMATH_TOKEN\b', ctx)),
            'mention_order': int(mention_order),
        })

    return out

def example_key(row: dict) -> str:
    sent = ' '.join(str(row['sent']).split())
    return f"{row['bibcode']}|||{sent}|||{int(row['start'])}|||{int(row['end'])}"

def hash_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


### Build master mention table
This extracts every cleaned ‘dark matter’ mention with a symmetric context window.

---

In [ ]:
if FORCE_REBUILD_MASTER or not MASTER_PARQUET.exists() or not MASTER_MANIFEST.exists():
    raw_pat = re.compile(r'\bdark\b.{0,20}\bmatter\b', re.IGNORECASE | re.DOTALL)
    df_sub = df.loc[
        df['abstract'].str.contains(raw_pat, na=False),
        ['source_row_index', 'year', 'bibcode', 'abstract']
    ].copy()

    master_rows = []
    for source_row_index, year, bibcode, text in tqdm(
        zip(df_sub['source_row_index'].values, df_sub['year'].values, df_sub['bibcode'].values, df_sub['abstract'].values),
        total=len(df_sub),
        desc='Extract master mentions'
    ):
        exs = build_examples_from_abstract_pm1(text, bibcode=str(bibcode))
        for ex in exs:
            ex['source_row_index'] = int(source_row_index)
            ex['year'] = int(year)
            ex['time_bin_current'] = assign_time_bin(int(year))
            ex['example_key'] = example_key(ex)
            master_rows.append(ex)

    master_df = pd.DataFrame(master_rows)

    if master_df.empty:
        raise ValueError('Master corpus build produced no mention rows.')

    master_df['sent'] = master_df['sent'].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()

    master_df = (
        master_df
        .sort_values(['source_row_index', 'mention_order', 'bibcode', 'start', 'end'], kind='mergesort')
        .drop_duplicates(subset=['example_key'], keep='first')
        .reset_index(drop=True)
    )
    master_df['example_id'] = np.arange(len(master_df), dtype=int)

    master_df = master_df[[
        'example_id', 'example_key', 'source_row_index', 'mention_order', 'year', 'time_bin_current',
        'bibcode', 'sent', 'start', 'end', 'has_math'
    ]].copy()

    master_df.to_parquet(MASTER_PARQUET, index=False)

    master_manifest = {
        'master_run_id': MASTER_RUN_ID,
        'source_parquet': workspace_relative(PARQUET_PATH),
        'source_parquet_sha256': hash_file(PARQUET_PATH),
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'output_parquet': workspace_relative(MASTER_PARQUET),
        'force_rebuild_master': FORCE_REBUILD_MASTER,
        'filters': {
            'require_year': True,
            'require_abstract': True,
            'lowercase_abstract': True,
            'excluded_arxiv_categories': bad_arx,
            'excluded_doctypes': bad_doc,
            'biology_filter_terms': sorted(biology_terms_strict),
            'earliest_backfill_year': EARLIEST_BACKFILL_YEAR,
            'end_year': END_YEAR,
        },
        'bin_schema_metadata': {
            'analysis_start_year': ANALYSIS_START_YEAR,
            'bin_years': BIN_YEARS,
            'end_year': END_YEAR,
            'time_bin_field_is_advisory_only': True,
        },
        'normalization_settings': {
            'target': TARGET,
            'max_ctx_words': MAX_CTX_WORDS,
            'min_ctx_words': MIN_CTX_WORDS,
            'min_center_sent_words': MIN_CENTER_SENT_WORDS,
            'soft_ctx_words': SOFT_CTX_WORDS,
            'max_dm_per_ctx': MAX_DM_PER_CTX,
            'max_math_tokens_per_ctx': MAX_MATH_TOKENS_PER_CTX,
        },
        'row_counts': {
            'filtered_source_rows': int(len(df)),
            'source_rows_with_dark_matter_pattern': int(len(df_sub)),
            'master_mentions': int(len(master_df)),
        },
        'master_parquet_sha256': hash_file(MASTER_PARQUET),
    }

    with open(MASTER_MANIFEST, 'w', encoding='utf-8') as f:
        json.dump(master_manifest, f, indent=2)

    print(f'Saved master corpus -> {MASTER_PARQUET}')
    print(f'Saved master manifest -> {MASTER_MANIFEST}')

master_df.head(5)


## Summary

---

In [ ]:
summary = {
    'n_mentions': len(master_df),
    'n_unique_bibcodes': int(master_df['bibcode'].nunique()),
    'year_min': int(master_df['year'].min()),
    'year_max': int(master_df['year'].max()),
    'has_math_rate': float(master_df['has_math'].mean()),
}
display(summary)

year_counts = master_df.groupby('year').size().rename('n_mentions').reset_index()
display(year_counts.head(20))


timebin_counts = (
    master_df.groupby('time_bin_current', dropna=False)
    .size()
    .rename('n_mentions')
    .reset_index()
    .sort_values('time_bin_current', kind='mergesort')
)
display(timebin_counts)
